In [5]:
import os
import requests
from openai.types.responses import ResponseTextDeltaEvent
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, SQLiteSession, OpenAIChatCompletionsModel, set_tracing_disabled
from IPython.display import Markdown, display

set_tracing_disabled(disabled=True)

ollama_client = AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
local_llm = OpenAIChatCompletionsModel(model="llama3.2", openai_client=ollama_client)

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

@function_tool
def push_tool(message: str) -> str:
    """Send the given message to the user as a push notification"""
    payload = {"user":pushover_user, "token": pushover_token, "message": message}
    response = requests.post(pushover_url, data=payload).status_code
    return f"Push notification sent with status code {response}"

notifier = Agent(
    name="Notifier",
    model=local_llm,
    instructions="You notify the user upon request",
    tools=[push_tool]
)

results =await Runner.run(notifier, "Notify the user that the task is complete")
print(results.final_output)

Task completed, and a push notification was successfully sent. The message conveyed to users that the task is now complete. Would you like to create another notification or perform some other action?
